In [56]:
import pandas as pd
from google.colab import files
import os

# Upload and find the correct filename
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename, encoding='latin-1')
if 'target' in df.columns:
    df = df.rename(columns={'target': 'label'})
display(df.head())

Saving combined_dataset.csv to combined_dataset (4).csv


,label,text
0,spam,Congratulations! You've been selected for a lu...
1,spam,URGENT: Your account has been compromised. Cli...
2,spam,You've won a free iPhone! Claim your prize by ...
3,spam,Act now and receive a 50% discount on all purc...
4,spam,Important notice: Your subscription will expir...


In [57]:
print(df['label'].value_counts())

label
ham     8555
spam    2406
Name: count, dtype: int64


In [58]:
# text preprocessing
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

keep_words = {
    "you", "your", "now", "free",
    "win", "prize", "click", "offer",
    "urgent", "call", "claim"
}

stop_words = stop_words - keep_words

def text_prepro(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)  # keep numbers

    words = text.split()
    words = [word for word in words if word not in stop_words]

    if not words:
        return ["empty"]

    return words


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [59]:
# Model code from scratch
import math # for log()
from collections import defaultdict # return 0 if key not found

class NaiveBayes:
    def __init__(self):
        self.word_counts = {
            "spam": defaultdict(int), # count one word in both spam and ham classes
            "ham": defaultdict(int)
        }
        self.class_counts = {"spam": 0, "ham": 0} # count  num of each classes
        self.total_words = {"spam": 0, "ham": 0} # count total words in each class
        self.vocab = set() # Stores all unique words

    def train(self, texts, labels):
        for text, label in zip(texts, labels):
            self.class_counts[label] += 1
            words = text_prepro(text)

            for word in words:
                self.word_counts[label][word] += 1
                self.total_words[label] += 1
                self.vocab.add(word)

    def word_prob(self, word, label):
        vocab_size = len(self.vocab)
        # Likelihood (with smoothing)
        return (self.word_counts[label][word] + 1) / (self.total_words[label] + vocab_size)

    def predict(self, text):
        words = text_prepro(text)
        total_docs = sum(self.class_counts.values())

        log_spam = math.log(self.class_counts["spam"] / total_docs)
        log_ham = math.log(self.class_counts["ham"] / total_docs)

        for word in words:
            log_spam += math.log(self.word_prob(word, "spam"))
            log_ham += math.log(self.word_prob(word, "ham"))

        threshold = 0.0

        # Compare probability of both classes
        return "spam" if (log_spam - log_ham) > threshold else "ham"

In [60]:
# Traing with scratch hand-written code without vectorizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report



X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

model = NaiveBayes()
model.train(X_train, y_train)

preds = [model.predict(text) for text in X_test]

model.train(df['text'], df['label'])

print("Prediction for spam or not :", model.predict(' Doston and Sardor also are going to with us, they want to buy some clothesfor wedding'))

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Prediction for spam or not : spam
Accuracy: 0.9357045143638851
              precision    recall  f1-score   support

         ham       0.98      0.94      0.96      1706
        spam       0.81      0.93      0.87       487

    accuracy                           0.94      2193
   macro avg       0.89      0.94      0.91      2193
weighted avg       0.94      0.94      0.94      2193



In [61]:
#Traing with sklear ready-to-use model, TfidVectorizer(giving more high weights to important words )

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
vectorizer = TfidfVectorizer()

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

preds = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, preds))
X = vectorizer.fit_transform(df['text'])
y = df['label']
model = MultinomialNB()
model.fit(X, y)
test = ["Doston and Sardor are going to buy clothes"]
test_vec = vectorizer.transform(test)
print(classification_report(y_test, preds))
print(model.predict(test_vec))

Accuracy: 0.8732330141358869
              precision    recall  f1-score   support

         ham       0.86      1.00      0.92      1706
        spam       1.00      0.43      0.60       487

    accuracy                           0.87      2193
   macro avg       0.93      0.71      0.76      2193
weighted avg       0.89      0.87      0.85      2193

['ham']


**Comparison  with Logistic Regression**

In [62]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)

lr_preds = lr_model.predict(X_test_vec)

In [64]:
from sklearn.metrics import classification_report, accuracy_score

print("=== Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
print("\n=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print(classification_report(y_test, lr_preds))

=== Naive Bayes ===
Accuracy: 0.8732330141358869
              precision    recall  f1-score   support

         ham       0.86      1.00      0.92      1706
        spam       1.00      0.43      0.60       487

    accuracy                           0.87      2193
   macro avg       0.93      0.71      0.76      2193
weighted avg       0.89      0.87      0.85      2193


=== Logistic Regression ===
Accuracy: 0.9407204742362061
              precision    recall  f1-score   support

         ham       0.94      0.99      0.96      1706
        spam       0.96      0.76      0.85       487

    accuracy                           0.94      2193
   macro avg       0.95      0.88      0.91      2193
weighted avg       0.94      0.94      0.94      2193

